# Speculative Decoding Walkthrough for Whispbook Voice Routing

This notebook explains speculative decoding, then runs the same gender/voice-routing task three ways:

1. a smaller draft model only,
2. a larger target model only,
3. speculative decoding with the larger model verified against tokens drafted by the smaller model.

The live cells are designed for macOS Apple Silicon and MLX. They are safe to read without running large models; set `RUN_LIVE_MLX_GENDER=1` before launching the kernel once your PDF path and checkpoints are ready.

## 1. Paper Context and Mechanism

The core idea comes from **Fast Inference from Transformers via Speculative Decoding** by Yaniv Leviathan, Matan Kalman, and Yossi Matias, and the closely related **Accelerating Large Language Model Decoding with Speculative Sampling** by Charlie Chen and co-authors.

Autoregressive decoding is usually serial: a large model produces token 1, then token 2, and so on. Speculative decoding changes the execution plan without changing the target model contract:

- a cheap draft model proposes `gamma` future tokens,
- the expensive target model verifies that drafted block in parallel,
- the accepted prefix is appended to the output,
- when a token is rejected, generation falls back to the target distribution at that point.

In this project, `gamma` is passed to MLX-LM as `num_draft_tokens`. The walkthrough fixes `PAPER_DEFAULT_GAMMA = 7`, matching the paper's trace example and several reported T5 translation settings, so the story/model matrix compares model and story effects without adding gamma as another variable. For this repo, the small and large Qwen checkpoints are tokenizer-compatible, which is required by the current MLX speculative path.

References: [Leviathan et al., arXiv:2211.17192](https://arxiv.org/abs/2211.17192), [Chen et al., arXiv:2302.01318](https://arxiv.org/abs/2302.01318).

![Speculative decoding flow](assets/speculative-decoding-flow.png)

## 2. Runtime Setup

VS Code notebook setup from the repository root:

```bash
uv sync
```

Then open this `.ipynb` in VS Code and select the project kernel, usually `.venv/bin/python`. The project dev environment includes `ipykernel`, `ipywidgets`, `nbformat`, `pandas`, `matplotlib`, `plotly`, and `mlx-lm==0.31.3`, so live MLX cells run in the selected VS Code kernel.

If VS Code does not list the project kernel, register it once:

```bash
uv run python -m ipykernel install --user --name whispbook-gender-specdec --display-name "Whispbook Gender SpecDec"
```

The notebook imports the local package from `src/`, so editable installation is not required when launched from this repository.

In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        package_dir = candidate / "src" / "whispbook_gender_specdec"
        if (candidate / "pyproject.toml").exists() and package_dir.exists():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from whispbook_gender_specdec.benchmark import (  # noqa: E402
    ModeConfig,
    default_mode_configs,
    load_builtin_story_samples_dataframe,
    load_story_benchmark_dataframe,
    normalize_sample_dataframe,
    run_benchmark_dataframe,
    source_hint_from_sample_row,
    write_benchmark_csv,
)
from whispbook_gender_specdec.checkpoints import discover_model_specs  # noqa: E402
from whispbook_gender_specdec.gender import GenderInferenceResult  # noqa: E402
from whispbook_gender_specdec.mlx_runner import infer_with_mlx  # noqa: E402
from whispbook_gender_specdec.results import (  # noqa: E402
    summarize_results_by_speaker_dataframe,
    summarize_results_dataframe,
    summarize_story_model_matrix_dataframe,
)
from whispbook_gender_specdec.utils.pdf_parser import (  # noqa: E402
    extract_chapter_text,
    parse_pdf_to_markdown,
)

print(f"Project root: {PROJECT_ROOT}")


## 3. Experiment Controls

Edit these values before running the live benchmark.

- `STORY_DIR`: local story benchmark folder; defaults to `data/story_2` in this working copy.
- `STORY_TEXT_PATH`: the full plain-text story to slice.
- `STORY_GROUND_TRUTH_PATH`: JSONL rows with sample text, speaker, and expected gender.
- `PDF_PATH`: optional fallback PDF to parse when the story dataset is missing.
- `TARGET_CHAPTER`: chapter number or heading token accepted by the project parser.
- `TARGET_POSITION`: start position inside the extracted chapter, measured in characters.
- `TARGET_LENGTH`: number of chapter characters to keep after `TARGET_POSITION`.
- `GROUND_TRUTH_BY_SPEAKER`: labels needed for accuracy on your own PDF. Without labels, the notebook still records predictions and timings, but accuracy is only meaningful for labeled samples.
- `PAPER_DEFAULT_GAMMA`: fixed paper-baseline number of draft tokens per speculative verification step.
- `MAX_TOKENS`: maximum generated JSON response tokens.
- `SMALL_MODEL_FAMILIES`: standalone small models to compare.
- `BIG_MODEL_FAMILIES`: standalone target models to compare, including two extra big models.
- `SPECULATIVE_MODEL_PAIRS`: explicit `(target, draft)` pairs for speculative decoding.
- `STORY_MATRIX_DIRS`: story fixtures included in the story/model matrix.
- `RUN_MODEL_GRID` and `RUN_STORY_MODEL_MATRIX`: run wider comparisons; these can download/load extra models.
- `MAX_TOKENS_GRID`: response-budget values for the fixed-gamma evolution plots.

In [ ]:
# Local story benchmark input. The notebook prefers this dataset when present.
STORY_DIR = PROJECT_ROOT / "data" / "story_2"
STORY_TEXT_PATH = next(iter(sorted(STORY_DIR.glob("*_20000_word_story.txt"))), STORY_DIR / "glass_archive_20000_word_story.txt")
STORY_METADATA_PATH = next(iter(sorted(STORY_DIR.glob("*_benchmark_metadata.json"))), STORY_DIR / "glass_archive_benchmark_metadata.json")
STORY_GROUND_TRUTH_PATH = next(iter(sorted(STORY_DIR.glob("*_speaker_attribution_ground_truth.jsonl"))), STORY_DIR / "glass_archive_speaker_attribution_ground_truth.jsonl")

# Optional PDF fallback. Used only when STORY_TEXT_PATH is missing.
PDF_PATH = PROJECT_ROOT / "data" / "your-own-book.pdf"

# Target extraction parameters: chapter x, position y, length z.
TARGET_CHAPTER = "1"
TARGET_POSITION = 0
TARGET_LENGTH = 22_000

# Model families from whispbook_gender_specdec.checkpoints.
SMALL_MODEL_FAMILIES = ["qwen_small", "gemma", "smollm"]
BIG_MODEL_FAMILIES = ["qwen_large", "llama_3_2_3b", "mistral_7b"]
SPECULATIVE_MODEL_PAIRS = [("qwen_large", "qwen_small")]

# Baseline trio used by the first live comparison and the fixed-gamma token-budget sweep.
SMALL_FAMILY = SMALL_MODEL_FAMILIES[0]
LARGE_FAMILY = BIG_MODEL_FAMILIES[0]
LOCAL_ONLY = False

# Inference knobs. Leave live inference disabled unless the kernel environment opts in.
RUN_LIVE = os.environ.get("RUN_LIVE_MLX_GENDER") == "1"
RUN_MODEL_GRID = True
RUN_STORY_MODEL_MATRIX = True
SAMPLE_LIMIT = 8
MAX_TOKENS = 64
PAPER_DEFAULT_GAMMA = 7
GAMMA = PAPER_DEFAULT_GAMMA

# Story/model matrix and token-budget knobs. Gamma stays fixed at PAPER_DEFAULT_GAMMA.
STORY_MATRIX_DIRS = sorted(path for path in (PROJECT_ROOT / "data").glob("story_*") if path.is_dir()) or [STORY_DIR]
MAX_TOKENS_GRID = [24, 48, 64]

# Fill this for your PDF if speaker names are known.
GROUND_TRUTH_BY_SPEAKER = {
    # "NORA": "female",
    # "HELMER": "male",
}

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_SUMMARY_CSV = OUTPUT_DIR / "model_comparison_summary.csv"
MODEL_COMPARISON_PREDICTIONS_CSV = OUTPUT_DIR / "model_comparison_predictions.csv"
STORY_MODEL_MATRIX_CSV = OUTPUT_DIR / "story_model_matrix_summary.csv"
STORY_MODEL_MATRIX_PREDICTIONS_CSV = OUTPUT_DIR / "story_model_matrix_predictions.csv"
SPECULATIVE_SWEEP_CSV = OUTPUT_DIR / "speculative_sweep.csv"
print(f"RUN_LIVE={RUN_LIVE}; set RUN_LIVE_MLX_GENDER=1 before launching Jupyter to run MLX inference.")

## 4. Load Benchmark Text and Labels

The notebook first looks for the local story fixture selected in the controls cell. It extracts `TARGET_CHAPTER` from the plain-text story, keeps the character window `[TARGET_POSITION:TARGET_POSITION + TARGET_LENGTH]`, then filters the JSONL ground-truth rows to that window. If the story files are missing, it falls back to the PDF path; if that is also missing, it uses the built-in public-domain samples.

From this point forward, candidate samples are carried as `samples_df`; benchmark outputs are DataFrames too.


In [ ]:
def slice_chapter_window(text: str, *, position: int, length: int | None) -> str:
    start = max(0, int(position))
    if length is None:
        return text[start:]
    return text[start : start + max(0, int(length))]


def find_story_chapter(text: str, chapter: str | int) -> tuple[str, int, int]:
    chapter_token = str(chapter).strip().lower()
    chapter_token = re.sub(r"^chapter\s+", "", chapter_token).strip()
    pattern = re.compile(r"(?im)^chapter\s+(?P<number>[a-z0-9ivxlcdm]+)\b.*$")
    matches = list(pattern.finditer(text))
    for index, match in enumerate(matches):
        if match.group("number").lower() != chapter_token:
            continue
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        return text[start:end].strip(), start, end
    raise ValueError(f"Chapter not found in story text: {chapter}")


def add_story_offsets(samples: pd.DataFrame, story_text: str) -> pd.DataFrame:
    enriched = samples.copy()
    starts: list[int] = []
    ends: list[int] = []
    cursor = 0
    for sample_text in enriched["text"].astype(str):
        start = story_text.find(sample_text, cursor)
        if start < 0:
            start = story_text.find(sample_text)
        starts.append(start)
        ends.append(start + len(sample_text) if start >= 0 else -1)
        if start >= 0:
            cursor = start + len(sample_text)
    enriched["start"] = starts
    enriched["end"] = ends
    return enriched


def select_window_samples(samples: pd.DataFrame, *, window_start: int, window_end: int) -> pd.DataFrame:
    selected = samples[
        (samples["start"] >= 0)
        & (samples["end"] > window_start)
        & (samples["start"] < window_end)
    ]
    if selected.empty:
        selected = samples
    return normalize_sample_dataframe(selected.head(SAMPLE_LIMIT).reset_index(drop=True))


def load_story_fixture() -> tuple[str, pd.DataFrame]:
    story_text = STORY_TEXT_PATH.read_text(encoding="utf-8")
    samples = load_story_benchmark_dataframe(
        ground_truth_path=STORY_GROUND_TRUTH_PATH,
        story_text_path=STORY_TEXT_PATH,
        metadata_path=STORY_METADATA_PATH if STORY_METADATA_PATH.exists() else None,
    )
    full_chapter_text, chapter_start, _chapter_end = find_story_chapter(story_text, TARGET_CHAPTER)
    chapter_text = slice_chapter_window(
        full_chapter_text,
        position=TARGET_POSITION,
        length=TARGET_LENGTH,
    )
    window_start = chapter_start + max(0, int(TARGET_POSITION))
    window_end = window_start + len(chapter_text)
    samples = add_story_offsets(samples, story_text)
    return chapter_text, select_window_samples(samples, window_start=window_start, window_end=window_end)


SPEAKER_LINE_RE = re.compile(r"^(?P<speaker>[A-Z][A-Z .'-]{1,48})[.:]\s+(?P<speech>.+)$")


def samples_from_chapter(chapter_text: str, *, source_title: str) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for index, line in enumerate(chapter_text.splitlines(), start=1):
        match = SPEAKER_LINE_RE.match(line.strip())
        if match is None:
            continue
        speaker = re.sub(r"\s+", " ", match.group("speaker").strip())
        text = f"{speaker}. {match.group('speech').strip()}"
        rows.append(
            {
                "sample_id": f"pdf_{index:04d}",
                "speaker": speaker,
                "expected_gender": GROUND_TRUTH_BY_SPEAKER.get(speaker, "unknown"),
                "text": text,
                "source_title": source_title,
                "source_author": "unknown",
                "source_url": str(PDF_PATH),
            }
        )
        if len(rows) >= SAMPLE_LIMIT:
            break

    if not rows:
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", chapter_text) if p.strip()]
        rows = [
            {
                "sample_id": f"pdf_chunk_{index:03d}",
                "speaker": "unknown",
                "expected_gender": "unknown",
                "text": paragraph[:1200],
                "source_title": source_title,
                "source_author": "unknown",
                "source_url": str(PDF_PATH),
            }
            for index, paragraph in enumerate(paragraphs[:SAMPLE_LIMIT], start=1)
        ]

    return normalize_sample_dataframe(pd.DataFrame(rows))


def load_target_text_or_builtin_fixture() -> tuple[str, pd.DataFrame, str]:
    if STORY_TEXT_PATH.exists() and STORY_GROUND_TRUTH_PATH.exists():
        chapter_text, samples = load_story_fixture()
        return chapter_text, samples, "story"

    if PDF_PATH.exists():
        markdown_path = OUTPUT_DIR / f"{PDF_PATH.stem}.md"
        markdown_text = parse_pdf_to_markdown(PDF_PATH, markdown_path)
        full_chapter_text = extract_chapter_text(markdown_text, TARGET_CHAPTER)
        chapter_text = slice_chapter_window(
            full_chapter_text,
            position=TARGET_POSITION,
            length=TARGET_LENGTH,
        )
        samples = samples_from_chapter(chapter_text, source_title=PDF_PATH.stem)
        return chapter_text, samples, "pdf"

    samples = load_builtin_story_samples_dataframe().head(SAMPLE_LIMIT).reset_index(drop=True)
    chapter_text = "\n".join(samples["text"].astype(str))
    return chapter_text, samples, "builtin"


chapter_text, samples_df, source_kind = load_target_text_or_builtin_fixture()
if source_kind == "story":
    display(Markdown(
        f"Loaded `{STORY_TEXT_PATH}` with labels from `{STORY_GROUND_TRUTH_PATH}`; "
        f"extracted chapter `{TARGET_CHAPTER}` from position `{TARGET_POSITION}` "
        f"for length `{TARGET_LENGTH}` into {len(samples_df)} candidate voice samples."
    ))
elif source_kind == "pdf":
    display(Markdown(
        f"Loaded `{PDF_PATH}` and extracted chapter `{TARGET_CHAPTER}` "
        f"from position `{TARGET_POSITION}` for length `{TARGET_LENGTH}` "
        f"into {len(samples_df)} candidate voice samples."
    ))
else:
    display(Markdown(f"`PDF_PATH` does not exist, so this run uses {len(samples_df)} built-in labeled samples from *A Doll's House*."))

print(chapter_text[:1400])


## 5. Inspect Candidate Voice Samples

Accuracy requires known labels. For a real PDF, fill `GROUND_TRUTH_BY_SPEAKER` above with the speaker names shown here, then rerun from the controls cell. The displayed object is the sample DataFrame that will be passed into every benchmark runner.


In [ ]:
def display_table(data: pd.DataFrame | list[dict[str, object]]) -> None:
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    display(frame)


sample_preview = samples_df.copy()
sample_preview["text"] = sample_preview["text"].astype(str).str.slice(0, 180)
preview_columns = [column for column in ["sample_id", "speaker", "expected_gender", "story_title", "source_title", "text"] if column in sample_preview.columns]
display_table(sample_preview.loc[:, preview_columns])


## 6. Discover Small and Large Checkpoints

`small_only` uses the draft-sized model alone. `large_only` uses the target model alone. `speculative` uses the large model as target and the small model as draft.

In [ ]:
manifest = discover_model_specs(local_only=LOCAL_ONLY)
small = manifest[SMALL_FAMILY]
large = manifest[LARGE_FAMILY]

modes = default_mode_configs(
    manifest,
    small_family=SMALL_FAMILY,
    large_family=LARGE_FAMILY,
)

selection_notes: list[str] = []


def standalone_mode_configs(families: list[str], *, prefix: str) -> list[ModeConfig]:
    configs: list[ModeConfig] = []
    seen: set[str] = set()
    for family in families:
        if family in seen:
            continue
        seen.add(family)
        spec = manifest.get(family)
        if spec is None:
            selection_notes.append(f"`{family}` is not declared in the checkpoint manifest.")
            continue
        if not spec.available or spec.model_ref is None:
            selection_notes.append(f"`{family}` is unavailable: {'; '.join(spec.notes)}")
            continue
        configs.append(ModeConfig(f"{prefix}:{family}", target=spec, draft=None))
    return configs


def speculative_pair_mode_configs(pairs: list[tuple[str, str]]) -> list[ModeConfig]:
    configs: list[ModeConfig] = []
    for target_family, draft_family in pairs:
        target = manifest.get(target_family)
        draft = manifest.get(draft_family)
        if target is None or draft is None:
            selection_notes.append(f"Skipping `{target_family}` + `{draft_family}`: missing manifest family.")
            continue
        if not target.available or target.model_ref is None:
            selection_notes.append(f"Skipping `{target_family}` target: {'; '.join(target.notes)}")
            continue
        if not draft.available or draft.model_ref is None:
            selection_notes.append(f"Skipping `{draft_family}` draft: {'; '.join(draft.notes)}")
            continue
        if target.vocab_size != draft.vocab_size:
            selection_notes.append(
                f"Skipping `{target_family}` + `{draft_family}` speculative pair: "
                f"vocab sizes differ ({target.vocab_size} != {draft.vocab_size})."
            )
            continue
        configs.append(ModeConfig(f"spec:{target_family}+{draft_family}", target=target, draft=draft))
    return configs


small_model_modes = standalone_mode_configs(SMALL_MODEL_FAMILIES, prefix="small")
big_model_modes = standalone_mode_configs(BIG_MODEL_FAMILIES, prefix="big")
model_comparison_modes = small_model_modes + big_model_modes
model_comparison_mode_lookup = {mode.name: mode for mode in model_comparison_modes}
speculative_pair_modes = speculative_pair_mode_configs(SPECULATIVE_MODEL_PAIRS)
speculative_pair_mode_lookup = {mode.name: mode for mode in speculative_pair_modes}

checkpoint_rows = [
    {
        "family": family,
        "label": spec.label,
        "source": spec.source,
        "available": spec.available,
        "model_ref": spec.model_ref,
        "vocab_size": spec.vocab_size,
        "notes": "; ".join(spec.notes),
    }
    for family, spec in manifest.items()
]
display_table(checkpoint_rows)
if selection_notes:
    display(Markdown("\n".join(f"- {note}" for note in selection_notes)))

if small.vocab_size != large.vocab_size:
    display(Markdown(f"Warning: `{SMALL_FAMILY}` and `{LARGE_FAMILY}` have different vocab sizes; speculative decoding will fail."))


## 7. Run the Three Inference Modes

This cell performs the real comparison and returns a full row-level results DataFrame. It also prints a summary DataFrame with accuracy, wall-clock time, JSON parse rate, and draft acceptance ratio. The project measures wall time around each call to `infer_with_mlx`, so results include the current runner's model-loading behavior. For production throughput, cache loaded models and tokenizers.


In [ ]:
def infer_one(mode: ModeConfig, sample: pd.Series, *, max_tokens: int, gamma: int) -> GenderInferenceResult:
    return infer_with_mlx(
        target=mode.target,
        draft=mode.draft,
        text=str(sample["text"]),
        source_hint=source_hint_from_sample_row(sample),
        max_tokens=max_tokens,
        num_draft_tokens=gamma,
    )


def run_live_comparison(*, max_tokens: int, gamma: int) -> pd.DataFrame:
    return run_benchmark_dataframe(
        samples_df,
        modes,
        lambda mode, sample: infer_one(mode, sample, max_tokens=max_tokens, gamma=gamma),
        parameters={"max_tokens": max_tokens, "num_draft_tokens": gamma},
    )


def summary_frame_to_dict(summary: pd.DataFrame) -> dict[str, dict[str, object]]:
    records = json.loads(summary.to_json(orient="records"))
    return {str(row.pop("mode")): row for row in records}


if not RUN_LIVE:
    display(Markdown("Live MLX inference is disabled. Relaunch Jupyter with `RUN_LIVE_MLX_GENDER=1` to run this cell."))
    predictions = pd.DataFrame()
else:
    predictions = run_live_comparison(max_tokens=MAX_TOKENS, gamma=GAMMA)
    summary = summarize_results_dataframe(predictions)
    print(json.dumps(summary_frame_to_dict(summary), indent=2))
    display_table(summary)
    display_table(predictions)


## 8. Minimal Single-Text DataFrame Run

This is the smallest useful entry point: define one text row, choose the small and large model families, set generation parameters, and get a row-level results DataFrame back. Use this cell when you want to sanity-check one prompt before running the full story fixture.


In [ ]:
SIMPLE_TEXT = "Mara Vale said, I will reopen the Archive before sunrise."
SIMPLE_SPEAKER = "Mara Vale"
SIMPLE_EXPECTED_GENDER = "female"
SIMPLE_SMALL_MODEL = "qwen_small"
SIMPLE_LARGE_MODEL = "qwen_large"
SIMPLE_PARAMETERS = {
    "max_tokens": 64,
    "num_draft_tokens": 3,
}

simple_input_df = normalize_sample_dataframe(
    pd.DataFrame(
        [
            {
                "sample_id": "simple_001",
                "speaker": SIMPLE_SPEAKER,
                "expected_gender": SIMPLE_EXPECTED_GENDER,
                "text": SIMPLE_TEXT,
                "story_title": "manual single-text check",
            }
        ]
    )
)

simple_modes = default_mode_configs(
    manifest,
    small_family=SIMPLE_SMALL_MODEL,
    large_family=SIMPLE_LARGE_MODEL,
)

if not RUN_LIVE:
    simple_results_df = pd.DataFrame()
    display(Markdown("Live MLX inference is disabled. Set `RUN_LIVE_MLX_GENDER=1` before launching Jupyter to populate `simple_results_df`."))
    display_table(simple_input_df)
else:
    simple_results_df = run_benchmark_dataframe(
        simple_input_df,
        simple_modes,
        lambda mode, sample: infer_with_mlx(
            target=mode.target,
            draft=mode.draft,
            text=str(sample["text"]),
            source_hint=source_hint_from_sample_row(sample),
            max_tokens=SIMPLE_PARAMETERS["max_tokens"],
            num_draft_tokens=SIMPLE_PARAMETERS["num_draft_tokens"],
        ),
        parameters={
            "small_model_name": SIMPLE_SMALL_MODEL,
            "large_model_name": SIMPLE_LARGE_MODEL,
            **SIMPLE_PARAMETERS,
        },
    )
    display_table(summarize_results_dataframe(simple_results_df))
    display_table(simple_results_df)


In [ ]:
# One-off configurable benchmark cell.
# Edit these values, then run this cell to get `one_off_results_df`.
ONE_OFF_TEXT = "Idris Noor said, I can read the street map better than the Archive can."
ONE_OFF_SPEAKER = "Idris Noor"
ONE_OFF_EXPECTED_GENDER = "male"
ONE_OFF_SMALL_MODEL = "qwen_small"
ONE_OFF_LARGE_MODEL = "qwen_large"
ONE_OFF_PARAMETERS = {
    "max_tokens": 64,
    "num_draft_tokens": 3,
}

one_off_input_df = normalize_sample_dataframe(
    pd.DataFrame(
        [
            {
                "sample_id": "one_off_001",
                "speaker": ONE_OFF_SPEAKER,
                "expected_gender": ONE_OFF_EXPECTED_GENDER,
                "text": ONE_OFF_TEXT,
                "story_title": "manual one-off check",
            }
        ]
    )
)

one_off_modes = default_mode_configs(
    manifest,
    small_family=ONE_OFF_SMALL_MODEL,
    large_family=ONE_OFF_LARGE_MODEL,
)

if not RUN_LIVE:
    one_off_results_df = pd.DataFrame()
    display(Markdown("Live MLX inference is disabled. Set `RUN_LIVE_MLX_GENDER=1` before launching Jupyter to populate `one_off_results_df`."))
    display_table(one_off_input_df)
else:
    one_off_results_df = run_benchmark_dataframe(
        one_off_input_df,
        one_off_modes,
        lambda mode, sample: infer_with_mlx(
            target=mode.target,
            draft=mode.draft,
            text=str(sample["text"]),
            source_hint=source_hint_from_sample_row(sample),
            max_tokens=ONE_OFF_PARAMETERS["max_tokens"],
            num_draft_tokens=ONE_OFF_PARAMETERS["num_draft_tokens"],
        ),
        parameters={
            "small_model_name": ONE_OFF_SMALL_MODEL,
            "large_model_name": ONE_OFF_LARGE_MODEL,
            **ONE_OFF_PARAMETERS,
        },
    )
    display_table(one_off_results_df)


## 9. Compare Different Small and Big Models

Edit `SMALL_MODEL_FAMILIES` and `BIG_MODEL_FAMILIES` in the controls cell to choose which standalone models to test. The defaults compare three small-ish models and three target models: Qwen 4B plus Llama 3.2 3B and Mistral 7B. Set `RUN_MODEL_GRID = True` before running this cell; the two extra big models may download on first use.

`SPECULATIVE_MODEL_PAIRS` is separate because speculative decoding requires tokenizer-compatible target and draft models. The default keeps the known Qwen pair.

This section saves fresh DataFrames to `notebooks/outputs/model_comparison_summary.csv` and `notebooks/outputs/model_comparison_predictions.csv` so the interactive charts can be reopened without rerunning the models.


In [ ]:
def comparison_summary_frame(
    summary: pd.DataFrame,
    mode_lookup: dict[str, ModeConfig],
    *,
    comparison_set: str,
) -> pd.DataFrame:
    if summary.empty:
        return pd.DataFrame()
    rows: list[dict[str, object]] = []
    for row in summary.to_dict("records"):
        mode_name = str(row["mode"])
        mode = mode_lookup[mode_name]
        rows.append(
            {
                "comparison_set": comparison_set,
                "mode": mode_name,
                "group": mode_name.split(":", 1)[0],
                "target_family": mode.target.family,
                "target_label": mode.target.label,
                "draft_family": mode.draft.family if mode.draft is not None else None,
                "source": mode.target.source,
                "vocab_size": mode.target.vocab_size,
                **{key: value for key, value in row.items() if key != "mode"},
            }
        )
    return pd.DataFrame(rows)


def comparison_prediction_frame(results: pd.DataFrame, *, comparison_set: str) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    frame = results.copy()
    frame.insert(0, "comparison_set", comparison_set)
    return frame


def save_csv_frame(path: Path, frame: pd.DataFrame) -> None:
    if frame.empty:
        return
    write_benchmark_csv(frame, path)
    display(Markdown(f"Saved `{path.relative_to(PROJECT_ROOT)}` with {len(frame)} rows."))


model_comparison_summary_frame = pd.DataFrame()
model_comparison_prediction_frame = pd.DataFrame()


if not RUN_LIVE:
    display(Markdown("Model grid skipped because live MLX inference is disabled."))
    model_comparison_results = pd.DataFrame()
    speculative_pair_results = pd.DataFrame()
elif not RUN_MODEL_GRID:
    display(Markdown("Model grid skipped. Set `RUN_MODEL_GRID = True` in the controls cell to run the selected small and big model comparisons."))
    model_comparison_results = pd.DataFrame()
    speculative_pair_results = pd.DataFrame()
else:
    model_comparison_results = run_benchmark_dataframe(
        samples_df,
        model_comparison_modes,
        lambda mode, sample: infer_one(mode, sample, max_tokens=MAX_TOKENS, gamma=GAMMA),
        parameters={"max_tokens": MAX_TOKENS, "num_draft_tokens": GAMMA},
    )
    model_comparison_summary_frame = comparison_summary_frame(
        summarize_results_dataframe(model_comparison_results),
        model_comparison_mode_lookup,
        comparison_set="standalone",
    )
    model_comparison_prediction_frame = comparison_prediction_frame(
        model_comparison_results,
        comparison_set="standalone",
    )
    display(Markdown("### Standalone small and big model comparison"))
    display_table(model_comparison_summary_frame)
    display_table(model_comparison_prediction_frame)

    if speculative_pair_modes:
        speculative_pair_results = run_benchmark_dataframe(
            samples_df,
            speculative_pair_modes,
            lambda mode, sample: infer_one(mode, sample, max_tokens=MAX_TOKENS, gamma=GAMMA),
            parameters={"max_tokens": MAX_TOKENS, "num_draft_tokens": GAMMA},
        )
        speculative_pair_summary_frame = comparison_summary_frame(
            summarize_results_dataframe(speculative_pair_results),
            speculative_pair_mode_lookup,
            comparison_set="speculative_pair",
        )
        speculative_pair_prediction_frame = comparison_prediction_frame(
            speculative_pair_results,
            comparison_set="speculative_pair",
        )
        model_comparison_summary_frame = pd.concat(
            [model_comparison_summary_frame, speculative_pair_summary_frame],
            ignore_index=True,
        )
        model_comparison_prediction_frame = pd.concat(
            [model_comparison_prediction_frame, speculative_pair_prediction_frame],
            ignore_index=True,
        )
        display(Markdown("### Explicit speculative pair comparison"))
        display_table(speculative_pair_summary_frame)
        display_table(speculative_pair_prediction_frame)
    else:
        speculative_pair_results = pd.DataFrame()
        display(Markdown("No compatible speculative pairs selected."))

    save_csv_frame(MODEL_COMPARISON_SUMMARY_CSV, model_comparison_summary_frame)
    save_csv_frame(MODEL_COMPARISON_PREDICTIONS_CSV, model_comparison_prediction_frame)


## 10. Interactive Model Comparison

Run this after the model-grid cell above. Use the legend to filter model groups or checkpoint sources, hover points for details, and use the dropdown chart to switch between accuracy, latency, JSON parse rate, and draft acceptance. If no in-memory results are available, this section loads `notebooks/outputs/model_comparison_summary.csv` as a DataFrame.


In [ ]:
def collect_model_comparison_frame() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    if "model_comparison_results" in globals() and isinstance(model_comparison_results, pd.DataFrame) and not model_comparison_results.empty:
        frames.append(
            comparison_summary_frame(
                summarize_results_dataframe(model_comparison_results),
                model_comparison_mode_lookup,
                comparison_set="standalone",
            )
        )
    if "speculative_pair_results" in globals() and isinstance(speculative_pair_results, pd.DataFrame) and not speculative_pair_results.empty:
        frames.append(
            comparison_summary_frame(
                summarize_results_dataframe(speculative_pair_results),
                speculative_pair_mode_lookup,
                comparison_set="speculative_pair",
            )
        )
    if not frames and "predictions" in globals() and isinstance(predictions, pd.DataFrame) and not predictions.empty:
        frames.append(
            comparison_summary_frame(
                summarize_results_dataframe(predictions),
                {mode.name: mode for mode in modes},
                comparison_set="baseline",
            )
        )
    if frames:
        return pd.concat(frames, ignore_index=True)

    cached = load_csv_frame(MODEL_COMPARISON_SUMMARY_CSV)
    if not cached.empty:
        display(Markdown(f"Loaded cached comparison summary from `{MODEL_COMPARISON_SUMMARY_CSV.relative_to(PROJECT_ROOT)}`. Rerun section 9 to refresh it."))
    return cached


def load_csv_frame(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    return frame.where(pd.notna(frame), None)


def show_plotly_figure(fig) -> None:
    try:
        fig.show()
        return
    except ValueError as error:
        if "nbformat" not in str(error):
            raise

    try:
        import nbformat
        import plotly.io._renderers as plotly_renderers
    except ImportError:
        display(Markdown(
            "Plotly inline rendering needs `nbformat` in the selected kernel. "
            "Run `uv sync --frozen`, restart the VS Code kernel, "
            "and rerun this cell."
        ))
    else:
        plotly_renderers.nbformat = nbformat
        try:
            fig.show()
            return
        except ValueError as error:
            if "nbformat" not in str(error):
                raise
            display(Markdown(
                "Plotly still cannot render inline in this running kernel. "
                "Restart the VS Code kernel after syncing dependencies. "
                "Opening the figure in a browser renderer for this run."
            ))

    try:
        fig.show(renderer="browser")
    except Exception as error:
        display(Markdown(f"Plotly browser fallback also failed: `{error}`."))


def plot_interactive_model_comparison(frame: pd.DataFrame) -> None:
    if frame.empty:
        display(Markdown("No model comparison rows yet. Run the model-grid cell, or set `RUN_MODEL_GRID = True` and rerun section 9."))
        return

    try:
        import plotly.express as px
        import plotly.graph_objects as go
    except ImportError:
        display(Markdown("Install `plotly` to render interactive comparison plots."))
        display_table(frame)
        return

    frame = frame.copy()
    column_defaults = {
        "comparison_set": "cached",
        "mode": "unknown",
        "group": None,
        "target_family": "unknown",
        "target_label": None,
        "draft_family": None,
        "source": "unknown",
        "vocab_size": 0,
        "total": 0,
        "correct": 0,
        "accuracy": 0.0,
        "parse_ok_rate": 0.0,
        "avg_wall_ms": 0.0,
        "avg_draft_acceptance_ratio": 0.0,
    }
    for column, default in column_defaults.items():
        if column not in frame.columns:
            frame[column] = default
    frame["group"] = frame["group"].fillna(frame["mode"].astype(str).str.split(":", n=1).str[0])
    for column in ["vocab_size", "total", "correct", "accuracy", "parse_ok_rate", "avg_wall_ms", "avg_draft_acceptance_ratio"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0)
    frame["draft_family"] = frame["draft_family"].fillna("none")
    frame["target_label"] = frame["target_label"].fillna(frame["target_family"])
    frame["source"] = frame["source"].fillna("unknown")
    frame["vocab_size"] = frame["vocab_size"].fillna(0).astype(int)
    frame["mode_label"] = frame["mode"].astype(str).str.replace(":", ": ", regex=False)
    frame = frame.sort_values(["group", "avg_wall_ms", "mode"])

    display_table(frame)

    scatter = px.scatter(
        frame,
        x="avg_wall_ms",
        y="accuracy",
        color="group",
        symbol="source",
        size="total",
        hover_name="mode_label",
        hover_data={
            "target_family": True,
            "target_label": True,
            "draft_family": True,
            "vocab_size": True,
            "correct": True,
            "total": True,
            "parse_ok_rate": ":.2%",
            "avg_draft_acceptance_ratio": ":.2%",
            "avg_wall_ms": ":.0f",
            "accuracy": ":.2%",
        },
        labels={
            "avg_wall_ms": "Average wall time (ms)",
            "accuracy": "Accuracy",
            "group": "Model group",
            "source": "Checkpoint source",
            "total": "Samples",
        },
        title="Model comparison: accuracy vs latency",
    )
    scatter.update_traces(marker={"line": {"width": 1, "color": "white"}})
    scatter.update_layout(height=520, legend_title_text="Click to filter", hovermode="closest")
    show_plotly_figure(scatter)

    metric_options = [
        ("accuracy", "Accuracy", ".2%"),
        ("avg_wall_ms", "Average wall time (ms)", ",.0f"),
        ("parse_ok_rate", "JSON parse rate", ".2%"),
        ("avg_draft_acceptance_ratio", "Draft acceptance ratio", ".2%"),
    ]
    group_colors = {
        "small": "#2563eb",
        "big": "#dc2626",
        "spec": "#16a34a",
        "small_only": "#2563eb",
        "large_only": "#dc2626",
        "speculative": "#16a34a",
    }
    customdata = frame[["target_family", "target_label", "draft_family", "source", "total", "correct"]].to_numpy()
    marker_color = frame["group"].map(group_colors).fillna("#64748b")

    bars = go.Figure()
    for index, (_metric, label, value_format) in enumerate(metric_options):
        bars.add_trace(
            go.Bar(
                x=frame["mode_label"],
                y=frame[_metric],
                marker_color=marker_color,
                customdata=customdata,
                name=label,
                visible=index == 0,
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    + f"{label}: %{{y:{value_format}}}<br>"
                    + "target: %{customdata[0]}<br>"
                    + "label: %{customdata[1]}<br>"
                    + "draft: %{customdata[2]}<br>"
                    + "source: %{customdata[3]}<br>"
                    + "correct: %{customdata[5]}/%{customdata[4]}"
                    + "<extra></extra>"
                ),
            )
        )

    buttons = []
    for index, (_metric, label, _value_format) in enumerate(metric_options):
        buttons.append(
            {
                "label": label,
                "method": "update",
                "args": [
                    {"visible": [trace_index == index for trace_index in range(len(metric_options))]},
                    {"title": f"Model comparison by {label.lower()}", "yaxis": {"title": label, "autorange": True}},
                ],
            }
        )

    bars.update_layout(
        title="Model comparison by accuracy",
        xaxis_title="Mode",
        yaxis_title="Accuracy",
        height=520,
        showlegend=False,
        updatemenus=[
            {
                "buttons": buttons,
                "direction": "down",
                "x": 1.0,
                "xanchor": "right",
                "y": 1.16,
                "yanchor": "top",
            }
        ],
    )
    bars.update_xaxes(tickangle=-25)
    show_plotly_figure(bars)


model_comparison_frame = collect_model_comparison_frame()
plot_interactive_model_comparison(model_comparison_frame)


## 11. Story x Small x Large Matrix

This run expands the comparison from one story to every fixture in `STORY_MATRIX_DIRS`. Each matrix point is a `(story, small_model, large_model)` combination with fixed `PAPER_DEFAULT_GAMMA`; gamma is recorded for traceability but never swept.

For each point, the notebook benchmarks the small model alone, the big model alone, and the speculative target+draft path only when that pair is tokenizer-compatible. The long-form summary keeps one row per mode, then the pivots and heatmaps expose model-pair matrices for accuracy, latency, and draft acceptance.

In [ ]:
def one_fixture_path(story_dir: Path, pattern: str) -> Path:
    matches = sorted(story_dir.glob(pattern))
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one {pattern} in {story_dir}, found {len(matches)}")
    return matches[0]


def load_matrix_story_samples(story_dir: Path) -> pd.DataFrame:
    text_path = one_fixture_path(story_dir, "*_20000_word_story.txt")
    metadata_path = one_fixture_path(story_dir, "*_benchmark_metadata.json")
    ground_truth_path = one_fixture_path(story_dir, "*_speaker_attribution_ground_truth.jsonl")
    story_samples = load_story_benchmark_dataframe(
        ground_truth_path=ground_truth_path,
        story_text_path=text_path,
        metadata_path=metadata_path,
    )
    if SAMPLE_LIMIT:
        story_samples = story_samples.head(SAMPLE_LIMIT)
    story_samples = story_samples.copy()
    story_samples["story_key"] = story_dir.name
    if "story_title" not in story_samples.columns:
        story_samples["story_title"] = story_dir.name
    story_samples["story_title"] = story_samples["story_title"].fillna(story_dir.name).astype(str)
    return story_samples


def story_model_pair_modes(small_family: str, large_family: str) -> tuple[list[ModeConfig], list[str]]:
    notes: list[str] = []
    pair_modes: list[ModeConfig] = []
    small_spec = manifest.get(small_family)
    large_spec = manifest.get(large_family)
    if small_spec is None:
        notes.append(f"Skipping `{small_family}`: missing manifest family.")
    elif not small_spec.available or small_spec.model_ref is None:
        notes.append(f"Skipping `{small_family}` standalone mode: {'; '.join(small_spec.notes)}")
    else:
        pair_modes.append(ModeConfig("small_only", target=small_spec, draft=None))

    if large_spec is None:
        notes.append(f"Skipping `{large_family}`: missing manifest family.")
    elif not large_spec.available or large_spec.model_ref is None:
        notes.append(f"Skipping `{large_family}` standalone mode: {'; '.join(large_spec.notes)}")
    else:
        pair_modes.append(ModeConfig("large_only", target=large_spec, draft=None))

    if small_spec is not None and large_spec is not None:
        compatible = (
            small_spec.available
            and large_spec.available
            and small_spec.model_ref is not None
            and large_spec.model_ref is not None
            and small_spec.vocab_size == large_spec.vocab_size
        )
        if compatible:
            pair_modes.append(ModeConfig("speculative", target=large_spec, draft=small_spec))
        else:
            notes.append(
                f"Skipping speculative `{large_family}` + `{small_family}`: "
                f"vocab sizes differ ({large_spec.vocab_size} != {small_spec.vocab_size}) or a model is unavailable."
            )
    return pair_modes, notes


def run_story_model_matrix() -> tuple[pd.DataFrame, list[str]]:
    frames: list[pd.DataFrame] = []
    notes: list[str] = []
    for story_dir in STORY_MATRIX_DIRS:
        story_samples = load_matrix_story_samples(story_dir)
        story_title = str(story_samples["story_title"].iloc[0]) if not story_samples.empty else story_dir.name
        for small_family in SMALL_MODEL_FAMILIES:
            for large_family in BIG_MODEL_FAMILIES:
                pair_modes, pair_notes = story_model_pair_modes(small_family, large_family)
                notes.extend(f"{story_dir.name}: {note}" for note in pair_notes)
                if not pair_modes:
                    continue
                point_results = run_benchmark_dataframe(
                    story_samples,
                    pair_modes,
                    lambda mode, sample: infer_one(
                        mode,
                        sample,
                        max_tokens=MAX_TOKENS,
                        gamma=PAPER_DEFAULT_GAMMA,
                    ),
                    parameters={
                        "small_model_name": small_family,
                        "large_model_name": large_family,
                        "max_tokens": MAX_TOKENS,
                        "num_draft_tokens": PAPER_DEFAULT_GAMMA,
                    },
                )
                frames.append(point_results)
                print(
                    f"finished {story_dir.name}: {small_family} x {large_family} "
                    f"on {len(story_samples)} samples with gamma={PAPER_DEFAULT_GAMMA}"
                )
    if not frames:
        return pd.DataFrame(), notes
    return pd.concat(frames, ignore_index=True), notes


STORY_MATRIX_HEATMAP_MODES = ["small_only", "large_only", "speculative"]


def pair_label(frame: pd.DataFrame) -> pd.Series:
    return frame["small_model_name"].astype(str) + " -> " + frame["large_model_name"].astype(str)


def display_story_model_matrix(summary: pd.DataFrame) -> None:
    if summary.empty:
        display(Markdown("No story/model matrix rows yet."))
        return

    frame = summary.copy()
    frame["pair"] = pair_label(frame)
    display_table(frame)

    pivot_specs = [
        ("accuracy", "Accuracy"),
        ("avg_wall_ms", "Average wall time (ms)"),
        ("avg_draft_acceptance_ratio", "Draft acceptance ratio"),
    ]
    for metric, label in pivot_specs:
        pivot = frame.pivot_table(
            index=["story_key", "story_title"],
            columns=["pair", "mode"],
            values=metric,
            aggfunc="mean",
        )
        display(Markdown(f"### {label} matrix"))
        display_table(pivot.reset_index())


def plot_story_model_heatmap(summary: pd.DataFrame, *, metric: str = "accuracy", mode: str = "speculative") -> None:
    if summary.empty:
        return
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return

    frame = summary[summary["mode"] == mode].copy()
    if frame.empty:
        display(Markdown(f"No `{mode}` rows are available for the heatmap."))
        return
    frame["pair"] = pair_label(frame)
    heatmap = frame.pivot_table(index="story_key", columns="pair", values=metric, aggfunc="mean").sort_index()
    if heatmap.empty:
        return

    fig_width = max(8, 1.4 * len(heatmap.columns))
    fig_height = max(4, 0.7 * len(heatmap.index))
    fig, axis = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
    image = axis.imshow(heatmap.to_numpy(dtype=float), aspect="auto", cmap="viridis")
    axis.set_title(f"{mode} {metric.replace('_', ' ')} by story and model pair")
    axis.set_xlabel("Small -> large model pair")
    axis.set_ylabel("Story")
    axis.set_xticks(range(len(heatmap.columns)), heatmap.columns, rotation=35, ha="right")
    axis.set_yticks(range(len(heatmap.index)), heatmap.index)
    for row_index, story_key in enumerate(heatmap.index):
        for column_index, pair in enumerate(heatmap.columns):
            value = heatmap.loc[story_key, pair]
            if pd.notna(value):
                axis.text(column_index, row_index, f"{value:.2f}", ha="center", va="center", color="white")
    fig.colorbar(image, ax=axis, label=metric.replace("_", " "))
    plt.show()


story_model_matrix_results = pd.DataFrame()
story_model_matrix_summary_frame = pd.DataFrame()
story_model_matrix_notes: list[str] = []

if not RUN_LIVE:
    display(Markdown("Story/model matrix skipped because live MLX inference is disabled."))
elif not RUN_STORY_MODEL_MATRIX:
    display(Markdown("Story/model matrix skipped. Set `RUN_STORY_MODEL_MATRIX = True` in the controls cell to run it."))
else:
    story_model_matrix_results, story_model_matrix_notes = run_story_model_matrix()
    story_model_matrix_summary_frame = summarize_story_model_matrix_dataframe(story_model_matrix_results)
    save_csv_frame(STORY_MODEL_MATRIX_PREDICTIONS_CSV, story_model_matrix_results)
    save_csv_frame(STORY_MODEL_MATRIX_CSV, story_model_matrix_summary_frame)

if story_model_matrix_summary_frame.empty:
    story_model_matrix_summary_frame = load_csv_frame(STORY_MODEL_MATRIX_CSV)
    if not story_model_matrix_summary_frame.empty:
        display(Markdown(f"Loaded cached story/model matrix from `{STORY_MODEL_MATRIX_CSV.relative_to(PROJECT_ROOT)}`. Rerun this section to refresh it."))

if story_model_matrix_notes:
    display(Markdown("\n".join(f"- {note}" for note in story_model_matrix_notes)))

display_story_model_matrix(story_model_matrix_summary_frame)
for heatmap_mode in STORY_MATRIX_HEATMAP_MODES:
    plot_story_model_heatmap(story_model_matrix_summary_frame, metric="accuracy", mode=heatmap_mode)

## 12. Fixed-Gamma `max_tokens` Sweep

This experiment keeps the task fixed and uses `PAPER_DEFAULT_GAMMA` for every speculative run. Only the response budget changes.

- `PAPER_DEFAULT_GAMMA` controls the fixed number of draft tokens per speculative verification step.
- `max_tokens` controls the response budget. If it is too low, the model may not finish valid JSON. If it is too high, latency can increase without improving accuracy on short classification tasks.

Fresh sweep rows are saved to `notebooks/outputs/speculative_sweep.csv` and the plot section can reload that CSV later.

In [ ]:
speculative_mode = [mode for mode in modes if mode.name == "speculative"][0]


def run_speculative_sweep() -> pd.DataFrame:
    sweep_rows: list[dict[str, object]] = []
    for max_tokens in MAX_TOKENS_GRID:
        started = time.perf_counter()
        point_results = run_benchmark_dataframe(
            samples_df,
            [speculative_mode],
            lambda mode, sample: infer_one(
                mode,
                sample,
                max_tokens=max_tokens,
                gamma=PAPER_DEFAULT_GAMMA,
            ),
            parameters={"max_tokens": max_tokens, "num_draft_tokens": PAPER_DEFAULT_GAMMA},
        )
        elapsed_ms = (time.perf_counter() - started) * 1000
        summary_row = summarize_results_dataframe(point_results).iloc[0]
        sweep_rows.append(
            {
                "mode": "speculative",
                "gamma": PAPER_DEFAULT_GAMMA,
                "max_tokens": max_tokens,
                "total_samples": int(summary_row["total"]),
                "accuracy": float(summary_row["accuracy"]),
                "avg_wall_ms": float(summary_row["avg_wall_ms"]),
                "parse_ok_rate": float(summary_row["parse_ok_rate"]),
                "avg_draft_acceptance_ratio": float(summary_row["avg_draft_acceptance_ratio"]),
                "sweep_wall_ms": elapsed_ms,
            }
        )
        print(f"finished max_tokens={max_tokens}, gamma={PAPER_DEFAULT_GAMMA}")
    return pd.DataFrame(sweep_rows)


if not RUN_LIVE:
    display(Markdown("Sweep skipped because live MLX inference is disabled."))
    sweep_frame = pd.DataFrame()
else:
    sweep_frame = run_speculative_sweep()
    save_csv_frame(SPECULATIVE_SWEEP_CSV, sweep_frame)
    display_table(sweep_frame)

## 13. Trace Metric Evolution

The plots show how accuracy, latency, and draft acceptance move as the response budget changes while gamma stays fixed at the paper-default setting. For a short JSON classification task, latency can get worse even when draft acceptance is high, because model-call overhead dominates.

In [ ]:
def plot_sweep(frame: pd.DataFrame) -> None:
    if frame.empty:
        frame = load_csv_frame(SPECULATIVE_SWEEP_CSV)
        if not frame.empty:
            display(Markdown(f"Loaded cached sweep results from `{SPECULATIVE_SWEEP_CSV.relative_to(PROJECT_ROOT)}`. Rerun section 12 to refresh them."))
        else:
            display(Markdown("No sweep rows to plot yet."))
            return

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        display(Markdown("Install `matplotlib` to render the evolution plots."))
        display_table(frame)
        return

    frame = frame.copy()
    for column in ["gamma", "max_tokens", "accuracy", "avg_wall_ms", "avg_draft_acceptance_ratio"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.sort_values("max_tokens")
    fixed_gamma = frame["gamma"].dropna().iloc[0] if frame["gamma"].notna().any() else PAPER_DEFAULT_GAMMA
    metrics = [
        ("accuracy", "Accuracy"),
        ("avg_wall_ms", "Avg wall time (ms)"),
        ("avg_draft_acceptance_ratio", "Draft acceptance ratio"),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4), constrained_layout=True)
    for axis, (metric, title) in zip(axes, metrics):
        axis.plot(frame["max_tokens"], frame[metric], marker="o", label=f"gamma={int(fixed_gamma)}")
        axis.set_title(title)
        axis.set_xlabel("max_tokens")
        axis.grid(True, alpha=0.3)
    axes[-1].legend(loc="best")
    plt.show()


plot_sweep(sweep_frame)

## 14. Reading the Results

Use this checklist when interpreting a run:

- The row-level benchmark output is a DataFrame: one row per `(mode, sample_id)` with model metadata, timings, raw response, parsed gender, correctness, and run parameters.
- The story/model matrix is a DataFrame rollup: one row per `(story_key, small_model_name, large_model_name, mode)` with fixed `PAPER_DEFAULT_GAMMA`.
- Summary and plotting cells consume DataFrames. CSV files are only persistence points; loading a CSV should produce a DataFrame before analysis.
- If `parse_ok_rate` drops, increase `MAX_TOKENS`; the model may be truncating the JSON response.
- If `avg_draft_acceptance_ratio` is low, the draft model is not predicting the target well for this prompt style; try a closer draft checkpoint.
- If speculative decoding is slower than `large_only`, the task is probably too short or model-loading overhead is dominating. This is common for compact JSON classification.
- If accuracy differs between `large_only` and `speculative`, inspect raw responses and labels first. Speculative decoding should preserve the target model distribution when the implementation and tokenizers are compatible, but small benchmark noise and parsing differences can still show up in this prototype.

For Whispbook, the most useful production metric is not a single sample latency; it is throughput over a chapter after models are loaded and prompts are batched or streamed through a stable runner.